In [28]:
import tkinter as tk
import os
from tkinter import filedialog, messagebox, font, simpledialog

root = tk.Tk()
root.title('메모장')
root.geometry('700x700')
current_file = None
is_modified = False

# frame 추가
frame = tk.Frame(root)
frame.pack(fill="both", expand=True)

# text를 root가 아니라 frame 위에 올림
text = tk.Text(
    frame,
    wrap="word",    # ← 단어 단위로 줄바꿈
    undo=True,      # ← 실행취소 기능 켜기
    font=("맑은 고딕", 11),  # ← 글꼴
    relief="flat",  # ← 테두리 스타일
    padx=6, pady=6  # ← 안쪽 여백
)
scrollbar = tk.Scrollbar(frame, command=text.yview)
text.configure(yscrollcommand=scrollbar.set)

scrollbar.pack(side="right", fill="y")
text.pack(fill="both", expand=True)

status = tk.Label(
    root, text="줄 1, 열 1",
    anchor="e", padx=8, relief="sunken"
)
status.pack(side="bottom", fill="x")

# ── 파일 작업 ────────────────────────────────────────

def new_file():
    global current_file, is_modified
    text.delete("1.0", "end")
    current_file = None
    is_modified = False
    root.title("메모장")

def open_file():
    global current_file, is_modified
    path = filedialog.askopenfilename(
        filetypes=[("텍스트 파일", "*.txt"), ("마크다운 파일", "*.md"), ("모든 파일", "*.*")]
    )
    if path:
        with open(path, encoding="utf-8") as f:
            text.delete("1.0", "end")
            text.insert("1.0", f.read())
        current_file = path
        root.title(f"{os.path.basename(path)} - 메모장")
        is_modified = False

def save_file():
    global current_file, is_modified
    if current_file:
        with open(current_file, "w", encoding="utf-8") as f:
            f.write(text.get("1.0", "end-1c"))
        is_modified = False
        title = root.title() # 제목에서 *제거
        if title.startswith("*"):
            root.title(title[1:])
    else:
        save_as()

def save_as():
    global current_file
    path = filedialog.asksaveasfilename(
        defaultextension=".txt",
        filetypes=[("텍스트 파일", "*.txt"), ("모든 파일", "*.*")]
    )
    if path:
        current_file = path
        save_file()
        root.title(f"{os.path.basename(path)} - 메모장")


# ── 편집 기능 ────────────────────────────────────────

def find_replace():
    win = tk.Toplevel(root)
    win.title("찾기/바꾸기")
    win.resizable(False, False)

    tk.Label(win, text="찾기:").grid(row=0, column=0, padx=8, pady=6, sticky="e")
    find_var = tk.StringVar()
    tk.Entry(win, textvariable=find_var, width=24).grid(row=0, column=1, padx=8)

    tk.Label(win, text="바꾸기:").grid(row=1, column=0, padx=8, pady=4, sticky="e")
    rep_var = tk.StringVar()
    tk.Entry(win, textvariable=rep_var, width=24).grid(row=1, column=1, padx=8)

    def do_replace_all():
        content = text.get("1.0", "end-1c")
        new = content.replace(find_var.get(), rep_var.get())
        text.delete("1.0", "end")
        text.insert("1.0", new)

    tk.Button(win, text="모두 바꾸기", command=do_replace_all).grid(
        row=2, column=0, columnspan=2, pady=8
    )

def choose_font():
    f = font.Font(font=text["font"])
    family = simpledialog.askstring("글꼴", "글꼴 이름:", initialvalue=f.actual()["family"])
    size   = simpledialog.askinteger("크기",  "크기:", initialvalue=f.actual()["size"])
    if family and size:
        text.configure(font=(family, size))

def toggle_wrap():
    text.configure(wrap="word" if word_wrap.get() else "none")

# ── 내부 유틸 ────────────────────────────────────────

def _update_status(_=None):
    pos = text.index("insert")
    line, col = pos.split(".")
    chars = len(text.get("1.0", "end-1c"))
    status.config(text=f"줄 {line}, 열 {int(col)+1}   문자 수: {chars}")


def _on_text_change(_=None):
    global is_modified
    if text.edit_modified():
        is_modified = True
        title = root.title()
        if not title.startswith("*"):
            root.title("*" + title)
        text.edit_modified(False)


def _check_save():
    global is_modified
    if is_modified:
        ans = messagebox.askyesnocancel("저장", "저장하지 않은 내용이 있습니다. 저장할까요?")
        if ans is True:
            save_file()
        elif ans is None:
            return True  # 취소
    return False

def on_close():
    if not _check_save():
        root.destroy()


# ── UI 구성 ─────────────────────────────────────────

menubar = tk.Menu(root)

# 파일
file_menu = tk.Menu(menubar, tearoff=0)
file_menu.add_command(label="새로 만들기", accelerator="Ctrl+N", command=new_file)
file_menu.add_command(label="열기...", accelerator="Ctrl+O", command=open_file)
file_menu.add_command(label="저장", accelerator="Ctrl+S", command=save_file)
file_menu.add_command(label="다른 이름으로 저장...", command=save_as)
file_menu.add_separator()
file_menu.add_command(label="종료", command=on_close)
menubar.add_cascade(label="파일", menu=file_menu)

# 편집
edit_menu = tk.Menu(menubar, tearoff=0)
edit_menu.add_command(label="실행 취소", accelerator="Ctrl+Z", command=lambda: text.edit_undo())
edit_menu.add_command(label="다시 실행", accelerator="Ctrl+Y", command=lambda: text.edit_redo())
edit_menu.add_separator()
edit_menu.add_command(label="잘라내기", accelerator="Ctrl+X", command=lambda: text.event_generate("<<Cut>>"))
edit_menu.add_command(label="복사", accelerator="Ctrl+C", command=lambda: text.event_generate("<<Copy>>"))
edit_menu.add_command(label="붙여넣기", accelerator="Ctrl+V", command=lambda: text.event_generate("<<Paste>>"))
edit_menu.add_command(label="모두 선택", accelerator="Ctrl+A", command=lambda: text.tag_add("sel", "1.0", "end"))
edit_menu.add_separator()
edit_menu.add_command(label="찾기/바꾸기...", accelerator="Ctrl+H", command=find_replace)
menubar.add_cascade(label="편집", menu=edit_menu)

# 서식
format_menu = tk.Menu(menubar, tearoff=0)
format_menu.add_command(label="글꼴...", command=choose_font)
word_wrap = tk.BooleanVar(value=True)
format_menu.add_checkbutton(label="자동 줄바꿈", variable=word_wrap, command=toggle_wrap)
menubar.add_cascade(label="서식", menu=format_menu)

# 단축키
root.bind("<Control-n>", lambda e: new_file())
root.bind("<Control-o>", lambda e: open_file())
root.bind("<Control-s>", lambda e: save_file())
root.bind("<Control-f>", lambda e: find_replace())

text.bind("<<Modified>>", _on_text_change)
text.bind("<KeyRelease>", _update_status)

root.protocol("WM_DELETE_WINDOW", on_close)  # 창 X버튼 눌렀을 때


root.config(menu=menubar)
root.mainloop()